# Chapter 14: Environment Simulation — Deterministic World-State

**Learning objectives:**
- Create `ToolEndpointSpec` objects with state transitions and failure modes
- Use `ClockControl` for time-dependent testing
- Manage `DataFixture` for reproducible test data
- Build and reset a deterministic `EnvironmentSimulator`

---
## Preamble

Chapters 11–13 introduced the adapter, gateway and policy engine — three layers that observe, intercept and govern tool calls. But the tools themselves remain fixed: `data_lookup` always returns the same JSON, `risk_check` always returns `"High"`. We controlled *whether* tools execute (gateway) and *whether they are permitted* (policy), but not *what they return*.

The **EnvironmentSimulator** closes this gap. It provides a deterministic, resettable world-state where each tool is backed by a `ToolEndpointSpec` — a formal specification with state transitions, failure modes and response templates. The same sequence of tool calls always produces the same state mutations and responses, enabling reproducible testing.

| Component | Purpose |
|---|---|
| `ToolEndpointSpec` | Defines a simulated tool with transitions and failures |
| `StateTransition` | Trigger condition → state mutation + response template |
| `FailureMode` | Named failure with probability and call-count trigger |
| `ClockControl` | Frozen, synthetic, or real-time modes |
| `DataFixture` | Versioned test data with SHA-256 integrity hash |
| `EnvironmentSimulator` | Deterministic world-state with reset and snapshot |

In [ ]:
# === Preamble (all notebooks) ===
import os
import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
from collections import Counter

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

In [ ]:
from importlib.resources import files
# Load the Chapter 4 store
from knowlytix.harness.testing import DOEGMSBenchmark, DOEHarnessConfig

config = DOEHarnessConfig(
    markdown_path=str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md")),
    stores_dir="../lab_stores",
    store_name="ch04_ground_truth",
    n_runs=4,
    ingest_epochs=200,
    device=str(DEVICE),
)
bench = DOEGMSBenchmark(config)
store = bench.ingest()
print(f"Store: {store.stats()['entities']} entities, "
      f"{store.stats()['triples']} triples, "
      f"{store.stats()['enm_entries']} ENM")

---

## ToolEndpointSpec: Defining Simulated Tools

A `ToolEndpointSpec` defines a single simulated tool endpoint with:

| Field | Type | Default | Purpose |
|---|---|---|---|
| `tool_name` | str | `""` | Name or fnmatch pattern |
| `description` | str | `""` | Human-readable description |
| `initial_state` | dict | `{}` | Per-endpoint initial state |
| `state_transitions` | list[StateTransition] | `[]` | Ordered; first match wins |
| `failure_modes` | list[FailureMode] | `[]` | Named failure modes |
| `default_response` | str | `"OK"` | Response when no transition matches |

In [ ]:
from knowlytix.harness.testing.environment import (
    EnvironmentSimulator, ToolEndpointSpec, StateTransition,
    FailureMode, ClockControl, DataFixture, ScheduledEvent,
)

# Define a create_ticket endpoint with state transitions
ticket_spec = ToolEndpointSpec(
    tool_name="create_ticket",
    description="Create a support ticket",
    initial_state={"tickets_created": 0, "last_ticket_id": ""},
    state_transitions=[
        StateTransition(
            trigger_action="*",
            state_mutation={
                "tickets_created": "{{state.tickets_created}}+1",
                "last_ticket_id": "TKT-{{args.title}}",
            },
            response_template="Ticket created: {{args.title}} [{{args.priority}}]",
        ),
    ],
    failure_modes=[
        FailureMode(
            mode_name="service_unavailable",
            error_message="Ticket service unavailable",
            probability=0.0,  # disabled for now
        ),
    ],
    default_response="Ticket created successfully",
)

print(f"Endpoint: {ticket_spec.tool_name}")
print(f"  transitions : {len(ticket_spec.state_transitions)}")
print(f"  failures    : {len(ticket_spec.failure_modes)}")
print(f"  initial     : {ticket_spec.initial_state}")
print(f"  default_resp: {ticket_spec.default_response!r}")

---

## StateTransition and FailureMode

**StateTransition** defines what happens when a tool call matches:

| Field | Type | Default | Purpose |
|---|---|---|---|
| `trigger_action` | str | `"*"` | fnmatch pattern for tool name |
| `trigger_condition` | str | `""` | Predicate on args (e.g. `"args.priority == high"`) |
| `state_mutation` | dict | `{}` | Key-value pairs merged into state |
| `response_template` | str | `""` | Template with `{{args.key}}` and `{{state.key}}` |

**FailureMode** defines probabilistic tool failures:

| Field | Type | Default | Purpose |
|---|---|---|---|
| `mode_name` | str | `""` | Human-readable identifier |
| `error_message` | str | `"Simulated failure"` | Error returned on failure |
| `probability` | float | `0.0` | Per-call failure probability [0, 1] |
| `trigger_after_n_calls` | int | `0` | If > 0, only fires after N calls |

In [ ]:
# Demonstrate state transitions and the template system
# Create a minimal environment with just the ticket endpoint
env = EnvironmentSimulator(tool_specs=[ticket_spec])

print("=== Initial State ===")
print(env.get_state())

# Call create_ticket
resp = env.handle_tool_call("create_ticket", {
    "title": "MDL-001-degradation",
    "priority": "high",
})
print(f"\nResponse: {resp!r}")
print("\n=== State After Call ===")
print(env.get_state())

# Second call
resp2 = env.handle_tool_call("create_ticket", {
    "title": "MDL-005-drift",
    "priority": "medium",
})
print(f"\nResponse: {resp2!r}")
print("\n=== State After 2 Calls ===")
state = env.get_state()
print(state)

In [ ]:
# Deterministic reset — same calls produce same state
env.reset(seed=42)

resp_replay = env.handle_tool_call("create_ticket", {
    "title": "MDL-001-degradation",
    "priority": "high",
})
resp2_replay = env.handle_tool_call("create_ticket", {
    "title": "MDL-005-drift",
    "priority": "medium",
})

print("=== Deterministic Replay ===")
print(f"Response 1 matches: {resp == resp_replay}")
print(f"Response 2 matches: {resp2 == resp2_replay}")
print(f"State matches:      {env.get_state() == state}")

---

## ClockControl: Deterministic Time

The `ClockControl` governs how `env.now()` behaves:

| Mode | Behavior |
|---|---|
| `"real"` | Wall-clock milliseconds |
| `"frozen"` | Always returns `frozen_at` timestamp |
| `"synthetic"` | Auto-increments by `tick_ms` each call |

**ScheduledEvent** fires when synthetic time reaches `at_ms`:

| Field | Type | Purpose |
|---|---|---|
| `at_ms` | int | Synthetic time trigger |
| `event_type` | str | `state_change`, `data_refresh`, or `fault_inject` |
| `payload` | dict | State mutations or fixture ID |

In [ ]:
# Frozen clock — time never advances
clock_frozen = ClockControl(mode="frozen", frozen_at="2026-01-15T10:00:00")
env_frozen = EnvironmentSimulator(tool_specs=[ticket_spec], clock=clock_frozen)

t1 = env_frozen.now()
t2 = env_frozen.now()
print(f"Frozen clock: t1={t1}, t2={t2}, same={t1 == t2}")

# Synthetic clock — auto-increments by tick_ms
clock_synth = ClockControl(mode="synthetic", tick_ms=1000)
env_synth = EnvironmentSimulator(tool_specs=[ticket_spec], clock=clock_synth)

times = [env_synth.now() for _ in range(5)]
print(f"\nSynthetic clock (tick_ms=1000): {times}")
print(f"Increment: {[times[i+1]-times[i] for i in range(len(times)-1)]}")

In [ ]:
# Scheduled events — fire at specific synthetic times
clock_events = ClockControl(
    mode="synthetic",
    tick_ms=100,
    scheduled_events=[
        ScheduledEvent(
            at_ms=500,
            event_type="state_change",
            payload={"maintenance_mode": True},
        ),
        ScheduledEvent(
            at_ms=1500,
            event_type="state_change",
            payload={"maintenance_mode": False},
        ),
    ],
)
env_events = EnvironmentSimulator(tool_specs=[ticket_spec], clock=clock_events)

# Advance clock in 500ms steps
print("=== Scheduled Events ===")
for step in range(4):
    fired = env_events.advance(500)
    maint = env_events.get_state().get("maintenance_mode", "not set")
    events_str = ", ".join(f"{e.event_type}@{e.at_ms}ms" for e in fired) or "(none)"
    print(f"  t={env_events._synthetic_time_ms:>5d}ms  "
          f"maintenance={str(maint):<6s}  fired: {events_str}")

---

## DataFixture: Versioned Test Data

A `DataFixture` bundles test data with a SHA-256 integrity hash:

| Field | Type | Default | Purpose |
|---|---|---|---|
| `fixture_id` | str | `""` | Unique identifier |
| `version` | int | `1` | Monotonic version number |
| `data` | dict | `{}` | The test data |
| `schema` | dict | `{}` | JSON Schema description |
| `content_hash` | str | `""` | SHA-256; auto-computed if empty |

In [ ]:
# Create a DataFixture with model inventory
fixture = DataFixture(
    fixture_id="model_inventory",
    version=1,
    data={
        "models": [
            {"id": "MDL-001", "name": "CreditScore-XGB", "risk": "High"},
            {"id": "MDL-002", "name": "FraudNet-LSTM", "risk": "High"},
            {"id": "MDL-003", "name": "AML-GBM", "risk": "High"},
            {"id": "MDL-004", "name": "MarketVaR-GARCH", "risk": "High"},
            {"id": "MDL-005", "name": "ChurnPredict-RF", "risk": "Medium"},
        ],
    },
)

print(f"Fixture: {fixture.fixture_id} v{fixture.version}")
print(f"Hash:    {fixture.content_hash[:32]}...")
print(f"Models:  {len(fixture.data['models'])}")
print(f"Integrity: {fixture.verify_integrity()}")

# Tamper detection
import copy
tampered = copy.deepcopy(fixture)
tampered.data["models"].append({"id": "MDL-999", "name": "Rogue", "risk": "Low"})
print(f"\nTampered integrity: {tampered.verify_integrity()}")

---

## Lab 14.1 — Build a Deterministic Test Environment

In this lab we use `create_mock_environment()` to build a full environment with four tool endpoints, a data fixture and a frozen clock. We then run the `FullMockHarness` through it and verify deterministic replay.

In [ ]:
# Lab 14.1a — Build environment and run through FullMockHarness
from knowlytix.harness.fixtures import create_mock_environment, FullMockHarness

env = create_mock_environment()

print("=== Environment Endpoints ===")
for spec in env.tool_specs:
    print(f"  {spec.tool_name:22s} transitions={len(spec.state_transitions)}  "
          f"failures={len(spec.failure_modes)}")

print(f"\nFixtures: {list(env.fixtures.keys())}")
print(f"Clock: mode={env.clock.mode}, frozen_at={env.clock.frozen_at}")
print("\n=== Initial State ===")
for k, v in sorted(env.get_state().items()):
    if not k.startswith("fixture:"):
        print(f"  {k}: {v}")

In [ ]:
from importlib.resources import files
# Lab 14.1b — Run a question through FullMockHarness with environment
harness = FullMockHarness(environment=env, recovery_mode="fallback")

context = open(str(files("knowlytix.harness.testing").joinpath("tests/model_risk_assessment.md"))).read()
prompt = (
    f"<report>\n{context}\n</report>\n\n"
    "Answer the following question using the report and any tools "
    "available to you.\n\n"
    "Create a ticket for MDL-001 degradation finding"
)
answer = harness.run(prompt)
print(f"Answer: {answer[:120]!r}")

# Check environment state after the run
print("\n=== State After Run ===")
state_after = env.get_state()
for k, v in sorted(state_after.items()):
    if not k.startswith("fixture:"):
        print(f"  {k}: {v}")

In [ ]:
# Lab 14.1c — Deterministic replay: reset and re-run
state_before_reset = env.get_state()
env.reset(seed=42)

harness2 = FullMockHarness(environment=env, recovery_mode="fallback")
answer2 = harness2.run(prompt)

state_replay = env.get_state()

print("=== Deterministic Replay ===")
print(f"Answer matches: {answer == answer2}")
print(f"State matches:  {state_replay == state_after}")
if answer != answer2:
    print(f"  Answer 1: {answer[:80]!r}")
    print(f"  Answer 2: {answer2[:80]!r}")

In [ ]:
# Lab 14.1d — State diff between initial and post-run
env.reset(seed=42)
state_init = env.get_state()

harness3 = FullMockHarness(environment=env, recovery_mode="fallback")
harness3.run(prompt)

state_post = env.get_state()
diff = env.get_state_diff(state_init, state_post)

print("=== State Diff ===")
print(f"Added keys  : {list(diff['added'].keys())}")
print(f"Removed keys: {list(diff['removed'].keys())}")
print("Changed keys:")
for k, v in diff['changed'].items():
    print(f"  {k}: {v['before']!r} -> {v['after']!r}")

In [ ]:
# Lab 14.1e — Snapshot and restore
env.reset(seed=42)
snap = env.snapshot_state()
print(f"Snapshot: {list(snap.keys())}")
print(f"  call_counts     : {snap['call_counts']}")
print(f"  synthetic_time  : {snap['synthetic_time_ms']}")

# Make some calls to mutate state
env.handle_tool_call("create_ticket", {"title": "test", "priority": "low"})
env.handle_tool_call("create_ticket", {"title": "test2", "priority": "high"})
print(f"\nAfter 2 calls: tickets_created={env.get_state().get('tickets_created')}")

# Restore to snapshot
env.restore_state(snap)
print(f"After restore: tickets_created={env.get_state().get('tickets_created')}")

---

## Lab 14.2 — Failure Mode Injection via Environment

The `FailureMode` on a `ToolEndpointSpec` allows probabilistic and call-count-triggered failures. When a failure triggers, `handle_tool_call()` returns `"ERROR: <message>"` instead of the normal response.

The key parameter is `trigger_after_n_calls`: when set to N, the failure can only activate after N calls to that tool. Combined with `probability`, this models degradation patterns like "database timeout after sustained load."

In [ ]:
# Lab 14.2a — Failure mode with trigger_after_n_calls
failing_spec = ToolEndpointSpec(
    tool_name="query_database",
    description="Query with degradation after 3 calls",
    initial_state={"query_count": 0},
    state_transitions=[
        StateTransition(
            trigger_action="*",
            response_template="Query result: 5 records for {{args.table}}",
        ),
    ],
    failure_modes=[
        FailureMode(
            mode_name="db_timeout",
            error_message="Database connection timed out",
            probability=1.0,  # 100% after trigger
            trigger_after_n_calls=3,
        ),
    ],
    default_response="Query executed",
)

env_fail = EnvironmentSimulator(tool_specs=[failing_spec])

print("=== Failure After N Calls ===")
for i in range(6):
    resp = env_fail.handle_tool_call("query_database", {"table": "models"})
    is_error = resp.startswith("ERROR")
    print(f"  Call {i+1}: {'FAIL' if is_error else 'OK':4s}  {resp[:50]}")

In [ ]:
# Lab 14.2b — Probabilistic failure mode

prob_spec = ToolEndpointSpec(
    tool_name="external_api",
    description="External API with 30% failure rate",
    state_transitions=[
        StateTransition(
            trigger_action="*",
            response_template="API response: success",
        ),
    ],
    failure_modes=[
        FailureMode(
            mode_name="intermittent",
            error_message="Service temporarily unavailable",
            probability=0.3,
        ),
    ],
)

env_prob = EnvironmentSimulator(tool_specs=[prob_spec])

# Run 100 calls to observe failure distribution
results = []
for _ in range(100):
    r = env_prob.handle_tool_call("external_api", {})
    results.append("ERROR" in r)

n_fail = sum(results)
print("=== Probabilistic Failure (100 calls, p=0.3) ===")
print(f"  Failures: {n_fail}/100 ({n_fail}%)")
print("  Expected: ~30 (within normal binomial range)")

In [ ]:
# Lab 14.2c — Agent recovery after environment failure
# Create environment where create_ticket fails after 2 calls
failing_ticket = ToolEndpointSpec(
    tool_name="create_ticket",
    description="Ticket service with degradation",
    initial_state={"tickets_created": 0},
    state_transitions=[
        StateTransition(
            trigger_action="*",
            state_mutation={"tickets_created": "{{state.tickets_created}}+1"},
            response_template="Ticket created: {{args.title}}",
        ),
    ],
    failure_modes=[
        FailureMode(
            mode_name="capacity_exceeded",
            error_message="Ticket system at capacity",
            probability=1.0,
            trigger_after_n_calls=2,
        ),
    ],
)

env_degrade = EnvironmentSimulator(
    tool_specs=[failing_ticket],
    clock=ClockControl(mode="frozen", frozen_at="2026-01-15T10:00:00"),
)

print("=== Tool Degradation Pattern ===")
for i in range(5):
    resp = env_degrade.handle_tool_call(
        "create_ticket", {"title": f"ticket-{i+1}", "priority": "high"}
    )
    status = "FAIL" if "ERROR" in resp else "OK"
    print(f"  Call {i+1}: [{status}] {resp[:60]}")

print(f"\nFinal state: tickets_created={env_degrade.get_state().get('tickets_created')}")

---

## YAML Serialization

The full environment configuration — tool specs, clock and fixtures — can be serialized to YAML for version-controlled, repeatable test environments.

In [ ]:
# YAML serialization of the environment configuration
import yaml

env_full = create_mock_environment()
yaml_data = {
    "environment": {
        "clock": env_full.clock.to_dict(),
        "fixtures": [f.to_dict() for f in env_full.fixtures.values()],
        "tool_specs": [s.to_dict() for s in env_full.tool_specs],
    }
}

yaml_str = yaml.dump(yaml_data, default_flow_style=False, sort_keys=False)
# Show first 40 lines
lines = yaml_str.split("\n")
print("=== Environment YAML (first 40 lines) ===")
print("\n".join(lines[:40]))
if len(lines) > 40:
    print(f"... ({len(lines) - 40} more lines)")

In [ ]:
# YAML round-trip: write to temp file, reload via from_yaml
import tempfile

with tempfile.NamedTemporaryFile(mode="w", suffix=".yaml", delete=False) as f:
    f.write(yaml_str)
    yaml_path = f.name

env_loaded = EnvironmentSimulator.from_yaml(yaml_path)
print(f"Loaded from YAML: {len(env_loaded.tool_specs)} endpoints, "
      f"{len(env_loaded.fixtures)} fixtures")
print(f"Clock: mode={env_loaded.clock.mode}, frozen_at={env_loaded.clock.frozen_at}")
for s in env_loaded.tool_specs:
    print(f"  {s.tool_name}")

os.unlink(yaml_path)

---

## End-to-End: 640 Questions with Environment Simulation

We run the full DOE benchmark through the `FullMockHarness` with the environment simulator enabled. Tools that have a simulated endpoint (like `create_ticket` and `send_notification`) are routed through the environment; tools without an endpoint (like `data_lookup` and `risk_check`) fall through to the harness's built-in handlers.

In [ ]:
# E2E: 640 questions with environment simulation
from knowlytix.harness.testing.adapters import AgentAdapter

e2e_env = create_mock_environment()
e2e_adapter = AgentAdapter(
    harness=FullMockHarness(environment=e2e_env, recovery_mode="fallback"),
)

doe_questions = bench.generate_questions()
traces = []
for dq in doe_questions:
    t = e2e_adapter.evaluate(
        question=dq.question.natural_language,
        context=context,
    )
    traces.append(t)

print(f"Collected {len(traces)} traces")
print(f"Errors   : {sum(1 for t in traces if t.error)}")
print(f"Non-empty: {sum(1 for t in traces if t.raw_output)}")

In [ ]:
# Analyze tool call distribution and environment state
all_tools = Counter(c.tool_name for t in traces for c in t.tool_calls)
print(f"=== Tool Call Distribution ({sum(all_tools.values())} total) ===")
for tool, count in all_tools.most_common():
    routed = "env" if e2e_env.has_endpoint(tool) else "harness"
    print(f"  {tool:22s} {count:>5d}  (routed to {routed})")

# Environment state after 640 questions
print("\n=== Environment State After 640 Questions ===")
final_state = e2e_env.get_state()
for k, v in sorted(final_state.items()):
    if not k.startswith("fixture:"):
        print(f"  {k}: {v!r}")

In [ ]:
# Latency statistics
latencies = [t.latency_ms for t in traces]
print("=== Latency (ms) ===")
print(f"  Mean   : {np.mean(latencies):.2f}")
print(f"  Std    : {np.std(latencies):.2f}")
print(f"  Median : {np.median(latencies):.2f}")
print(f"  P99    : {np.percentile(latencies, 99):.2f}")
print(f"  Max    : {np.max(latencies):.2f}")

---

## Exercises

**Exercise 14.1** — Create a custom `ToolEndpointSpec` for a `risk_assessment` tool with two state transitions: one that returns `"High"` when `args.model in [MDL-001, MDL-002]` and another that returns `"Medium"` otherwise. Verify both paths produce the correct responses.

**Exercise 14.2** — Configure a `ClockControl` in `"synthetic"` mode with a `ScheduledEvent` at `at_ms=2000` that sets `"system_degraded": True`. Advance the clock in 500ms steps and verify the event fires at the correct time.

**Exercise 14.3** — Run the 640-question benchmark with a `FailureMode` that triggers 50% failures on `send_notification` after 100 calls. Compare the tool call distribution against the baseline (no failures) and measure how many traces include error strings in their output.

---

## Summary

This chapter introduced the **EnvironmentSimulator** — the deterministic world-state layer that completes the black-box testing stack:

1. **ToolEndpointSpec** defines a simulated tool endpoint with state transitions, failure modes and response templates. The template system supports `{{args.key}}` and `{{state.key}}` placeholders for dynamic responses.

2. **StateTransition** maps trigger conditions to state mutations. Transitions are matched in order; the first match wins. The environment state accumulates across calls, enabling stateful tool simulation.

3. **FailureMode** injects probabilistic and call-count-triggered failures. Combined with `trigger_after_n_calls`, this models real degradation patterns like database timeouts under sustained load.

4. **ClockControl** provides three time modes — real, frozen and synthetic — plus scheduled events that fire at specific synthetic times for temporal scenario testing.

5. **DataFixture** bundles versioned test data with SHA-256 integrity hashes. Tampering is detected automatically via `verify_integrity()`.

6. **Deterministic replay** — `reset(seed=42)` restores the environment to its initial state with the same RNG seed. The same sequence of tool calls always produces the same responses and state mutations.

In the next chapter we add **Oracles** — composite judges that use GMS geometric primitives to evaluate agent behavior against expected outcomes, tool-call sequences and side effects.